In [ ]:
# ============================================================
# CELL 1 — Upload files
# ============================================================
from google.colab import files
uploaded = files.upload()
# Upload: orders.csv, restaurant.csv, menu.csv, food.csv, users.csv

# ============================================================


Saving restaurant.csv to restaurant.csv
Saving users.csv to users.csv
Saving food.csv to food.csv
Saving menu.csv to menu.csv
Saving orders.csv to orders.csv


In [ ]:
# CELL 2 — Load all tables
# ============================================================
import pandas as pd

orders      = pd.read_csv('orders.csv')
restaurant  = pd.read_csv('restaurant.csv')
menu        = pd.read_csv('menu.csv')
food        = pd.read_csv('food.csv')
users       = pd.read_csv('users.csv')

print("orders:", orders.shape)
print("restaurant:", restaurant.shape)
print("menu:", menu.shape)
print("food:", food.shape)
print("users:", users.shape)

/tmp/ipykernel_4308/476762550.py:7: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  menu        = pd.read_csv('menu.csv')


orders: (150281, 7)
restaurant: (148541, 12)
menu: (1179936, 6)
food: (371561, 4)
users: (100000, 12)


In [ ]:
# ============================================================
# CELL 3 — Clean restaurant ratings
# ============================================================
restaurant['rating'] = pd.to_numeric(
    restaurant['rating'].replace({'--': None, 'Too Few Ratings': None}),
    errors='coerce'
)
# Clean cost column (remove ₹ and spaces)
restaurant['cost'] = restaurant['cost'].str.replace('₹', '').str.replace(',', '').str.strip()
restaurant['cost'] = pd.to_numeric(restaurant['cost'], errors='coerce')



In [ ]:
# ============================================================
# CELL 4 — Derive context features from order_date
# ============================================================
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders['day_of_week']  = orders['order_date'].dt.day_name()        # Monday, Tuesday...
orders['is_weekend']   = orders['order_date'].dt.dayofweek >= 5    # True/False
orders['month']        = orders['order_date'].dt.month
orders['season']       = orders['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Monsoon', 10: 'Monsoon', 11: 'Monsoon'
})

In [ ]:
# ============================================================
# CELL 5 — Join all tables into one flat feature table
# ============================================================

# Fix: clean price column before aggregating
menu['price'] = pd.to_numeric(menu['price'], errors='coerce')

# Step 1: orders + restaurant
df = orders.merge(
    restaurant[['id', 'name', 'city', 'rating', 'cost', 'cuisine']],
    left_on='r_id', right_on='id', how='left'
).rename(columns={
    'name': 'restaurant_name',
    'rating': 'restaurant_rating',
    'cost': 'avg_cost_for_two',
    'cuisine': 'restaurant_cuisine'
})

# Step 2: + users
df = df.merge(
    users[['user_id', 'Age', 'Gender', 'Marital Status',
           'Occupation', 'Monthly Income', 'Educational Qualifications', 'Family size']],
    on='user_id', how='left'
)

# Step 3: + menu aggregated per restaurant
menu_food = menu.merge(food, on='f_id', how='left')

restaurant_menu_agg = menu_food.groupby('r_id').agg(
    menu_avg_price    = ('price', 'mean'),
    menu_item_count   = ('f_id', 'nunique'),
    veg_item_count    = ('veg_or_non_veg', lambda x: (x == 'Veg').sum()),
    nonveg_item_count = ('veg_or_non_veg', lambda x: (x == 'Non-veg').sum()),
    sample_items      = ('item', lambda x: ', '.join(x.dropna().unique()[:5]))
).reset_index()

df = df.merge(restaurant_menu_agg, on='r_id', how='left')

print("Final flat table shape:", df.shape)
print(df.columns.tolist())

Final flat table shape: (150281, 29)
['Unnamed: 0', 'order_date', 'sales_qty', 'sales_amount', 'currency', 'user_id', 'r_id', 'day_of_week', 'is_weekend', 'month', 'season', 'id', 'restaurant_name', 'city', 'restaurant_rating', 'avg_cost_for_two', 'restaurant_cuisine', 'Age', 'Gender', 'Marital Status', 'Occupation', 'Monthly Income', 'Educational Qualifications', 'Family size', 'menu_avg_price', 'menu_item_count', 'veg_item_count', 'nonveg_item_count', 'sample_items']


In [ ]:
# ============================================================
# CELL 6 — Preview
# ============================================================
df.head(3)


,Unnamed: 0,order_date,sales_qty,sales_amount,currency,user_id,r_id,day_of_week,is_weekend,month,...,Marital Status,Occupation,Monthly Income,Educational Qualifications,Family size,menu_avg_price,menu_item_count,veg_item_count,nonveg_item_count,sample_items
0,0,2017-10-10,100,41241,INR,49226,567335.0,Tuesday,False,10,...,Married,Self Employeed,25001 to 50000,Graduate,6,120.496183,131.0,100.0,31.0,"Aloo Tikki Burger, Veg Creamy Burger, Cheese B..."
1,1,2018-05-08,3,-1,INR,77359,531342.0,Tuesday,False,5,...,Single,Student,More than 50000,Post Graduate,3,128.112500,220.0,195.0,25.0,"Noodles, Pav Bhaji, Oreo Shake, Banana Shake, ..."
2,2,2018-04-06,1,875,INR,5321,158203.0,Friday,False,4,...,Married,Employee,More than 50000,Post Graduate,3,204.207792,195.0,158.0,73.0,"Aloo Tikki Burger, Paneer Tikka Sandwich, Oreo..."


In [ ]:
# ============================================================
# CELL 7 — Save
# ============================================================
df.to_csv('zomato_flat_features.csv', index=False)
files.download('zomato_flat_features.csv')
print("Saved! Shape:", df.shape)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved! Shape: (150281, 29)
